## Objective: 
### Use Subqueries, CTEs, and Window Functions to analyze sales data from the Superstore dataset. 

#### Step 1: Setup Data

In [0]:
create or replace table workspace.default.customers( cid string, cname string, segment string, country string, city string, state string, ps_code string, region string);

In [0]:
create or replace table workspace.default.products (pid string, category string,
sub_category string, pname string);

In [0]:
create or replace table workspace.default.orders( row_id int, oid string, order_date date, ship_date date, ship_mode string, cid string, pid string, sales double, quantity int, discount double, profit double)

In [0]:
insert into workspace.default.customers
select distinct `Customer ID` as cid, `Customer Name` as cname, Segment as segment, Country as country, City as city, State as state, `Postal Code` as ps_code, Region as region
from workspace.default.superstore_raw;

num_affected_rows,num_inserted_rows
4910,4910


In [0]:
insert into workspace.default.products
select distinct `Product ID` as pid, Category as category, `Sub-Category` as sub_category, `Product Name` as pname
from workspace.default.superstore_raw;

num_affected_rows,num_inserted_rows
1894,1894


In [0]:
insert into workspace.default.orders
select distinct `Row ID` as row_id, `Order ID` as oid, `Order Date` as order_date, `Ship Date` as ship_date, `Ship Mode` as ship_mode, `Customer ID` as cid, `Product ID` as pid, Sales as sales, Quantity as quantity, Discount as discount, Profit as profit
from workspace.default.superstore_raw;

num_affected_rows,num_inserted_rows
9994,9994


#### Step 2: Perform Required Queries

In [0]:
--Find all orders where sales are greater than the average sales. (Subquery)  
select *,(select round(avg(sales),2) from orders) as avg_sales from orders where sales >(select avg(sales) from orders);

row_id,oid,order_date,ship_date,ship_mode,cid,pid,sales,quantity,discount,profit,avg_sales
86,CA-2017-140088,2017-05-28,2017-05-30,Second Class,PO-18865,FUR-CH-10000863,301.96,2,0.0,33.2156,229.86
118,CA-2015-110457,2015-03-02,2015-03-06,Standard Class,DK-13090,FUR-TA-10001768,787.53,3,0.0,165.3813,229.86
150,CA-2016-114489,2016-12-05,2016-12-09,Standard Class,JE-16165,FUR-CH-10000454,1951.84,8,0.0,585.552,229.86
219,CA-2016-130162,2016-10-28,2016-11-01,Standard Class,JH-15910,TEC-PH-10002563,302.376,3,0.2,22.6782,229.86
224,CA-2015-169397,2015-12-24,2015-12-27,First Class,JB-15925,TEC-MA-10001148,479.988,4,0.7,-383.9904,229.86
236,US-2017-100930,2017-04-07,2017-04-12,Standard Class,CS-12400,TEC-AC-10003832,617.976,3,0.2,-7.7247,229.86
254,CA-2016-146941,2016-12-10,2016-12-13,First Class,DL-13315,OFF-EN-10003296,361.92,4,0.0,162.864,229.86
339,CA-2014-129924,2014-07-12,2014-07-17,Standard Class,AC-10420,FUR-TA-10004575,698.352,3,0.2,-17.4588,229.86
355,CA-2016-138520,2016-04-08,2016-04-13,Standard Class,JL-15505,FUR-BO-10002268,388.704,6,0.2,-4.8588,229.86
376,US-2014-119137,2014-07-23,2014-07-27,Standard Class,AG-10900,TEC-AC-10002076,479.04,10,0.2,-29.94,229.86


In [0]:

--Find the highest sales order for each customer. (Subquery)
select cid,max(sales) as Highest_Sales from orders group by cid order by cid;

cid,Highest_Sales
AA-10315,3930.072
AA-10375,499.98
AA-10480,479.97
AA-10645,1323.9
AB-10015,341.96
AB-10060,4355.168
AB-10105,9892.74
AB-10150,479.97
AB-10165,304.23
AB-10255,479.984


In [0]:
--calculate total sales for each customer

select cid as customer_id, round(sum(sales), 2) as total_sales from orders
group by cid order by cid;


customer_id,total_sales
AA-10315,5563.56
AA-10375,1056.39
AA-10480,1790.51
AA-10645,5086.93
AB-10015,886.16
AB-10060,7755.62
AB-10105,14473.57
AB-10150,966.71
AB-10165,1113.84
AB-10255,914.53


In [0]:
--Find customers whose total sales are above average. (CTE + Subquery)  
with customer_sales as ( select cid, round(sum(sales),2) AS total_sales from orders group by cid )

select cid, total_sales,(select round(avg(sales),2) from orders) as avg_sales from customer_sales where total_sales > (select round(avg(sales),2) from orders) order by cid;

cid,total_sales,avg_sales
AA-10315,5563.56,229.86
AA-10375,1056.39,229.86
AA-10480,1790.51,229.86
AA-10645,5086.93,229.86
AB-10015,886.16,229.86
AB-10060,7755.62,229.86
AB-10105,14473.57,229.86
AB-10150,966.71,229.86
AB-10165,1113.84,229.86
AB-10255,914.53,229.86


In [0]:
--Rank all customers based on total sales. (Window Function)
with customer_sales as ( select cid, round(sum(sales),2) AS total_sales from orders group by cid )

select cid,total_sales,rank() over (order by total_sales desc) as rnk from customer_sales  ;

cid,total_sales,rnk
SM-20320,25043.05,1
TC-20980,19052.22,2
RB-19360,15117.34,3
TA-21385,14595.62,4
AB-10105,14473.57,5
KL-16645,14175.23,6
SC-20095,14142.33,7
HL-15040,12873.3,8
SE-20110,12209.44,9
CC-12370,12129.07,10


In [0]:
--Assign row numbers to each order within a customer. (Window Function + PARTITION BY)  
select oid,cid,pid,row_number() over(partition by cid order by oid) as row_number from orders;

oid,cid,pid,row_number
CA-2014-128055,AA-10315,OFF-AP-10002765,1
CA-2014-128055,AA-10315,OFF-BI-10004390,2
CA-2014-138100,AA-10315,FUR-FU-10002456,3
CA-2014-138100,AA-10315,OFF-PA-10000349,4
CA-2015-121391,AA-10315,OFF-ST-10001590,5
CA-2016-103982,AA-10315,TEC-AC-10002857,6
CA-2016-103982,AA-10315,OFF-FA-10001332,7
CA-2016-103982,AA-10315,TEC-PH-10000895,8
CA-2016-103982,AA-10315,OFF-SU-10000151,9
CA-2017-147039,AA-10315,OFF-AP-10000576,10


In [0]:
--Display top 3 customers based on total sales. (Window Function)
with customer_sales as ( select cid,round(sum(sales),2) as total_sales, row_number() over(order by sum(sales) desc) as rnk from orders group by cid )

select cid,total_sales from customer_sales where rnk<=3


cid,total_sales
SM-20320,25043.05
TC-20980,19052.22
RB-19360,15117.34


#### Step 3: Final Combined Query

In [0]:
--Write one final query that shows: Customer Name  Total Sales  Rank  (Use JOIN + CTE + Window Function together) 
with customer_sales as(
    select cid,round(sum(sales),2) as total_sales,rank() over(order by sum(sales) desc) as Ranks from orders group by cid
),
customer_info as ( select distinct cid,cname from customers )

select c.cid,c.cname,cs.total_sales,cs.Ranks from customer_info c inner join customer_sales cs using(cid)


cid,cname,total_sales,Ranks
SM-20320,Sean Miller,25043.05,1
TC-20980,Tamara Chand,19052.22,2
RB-19360,Raymond Buch,15117.34,3
TA-21385,Tom Ashbrook,14595.62,4
AB-10105,Adrian Barton,14473.57,5
KL-16645,Ken Lonsdale,14175.23,6
SC-20095,Sanjit Chand,14142.33,7
HL-15040,Hunter Lopez,12873.3,8
SE-20110,Sanjit Engle,12209.44,9
CC-12370,Christopher Conant,12129.07,10


# Mini Project: Customer Sales Insights 

In [0]:
--Who are the top 5 customers?  
select cid,round(sum(sales),2) as Total_Sales from orders  group by cid order by Total_Sales desc limit 5

cid,Total_Sales
SM-20320,25043.05
TC-20980,19052.22
RB-19360,15117.34
TA-21385,14595.62
AB-10105,14473.57


In [0]:
--Who are the bottom 5 customers?
select cid,round(sum(sales),2) as Total_Sales from orders  group by cid order by Total_Sales asc limit 5

cid,Total_Sales
TS-21085,4.83
LD-16855,5.3
CJ-11875,16.52
MG-18205,16.74
RS-19870,22.33


In [0]:
--Which customers made only one order?
select cid,count(*) as order_placed from orders group by cid having order_placed=1;

cid,order_placed
LD-16855,1
AO-10810,1
CJ-11875,1
JR-15700,1
RE-19405,1


In [0]:
--Which customers have above-average sales? 
select cid,round(sum(sales),2) as total_sales,(select round(avg(sales),2) from orders) as avg_sales from orders group by cid 
having sum(sales)> (select avg(sales) as avg_sale from orders);

cid,total_sales,avg_sales
SK-19990,883.41,229.86
MS-17365,7443.69,229.86
JK-15205,4427.14,229.86
ML-17755,2071.91,229.86
BD-11770,658.47,229.86
JE-15715,8697.84,229.86
HG-14845,785.63,229.86
GH-14410,2819.47,229.86
VT-21700,1736.6,229.86
LS-17245,1008.14,229.86


In [0]:
--What is the highest order value per customer? 
select cid,max(sales) as max_sales from orders group by cid order by cid

cid,max_sales
AA-10315,3930.072
AA-10375,499.98
AA-10480,479.97
AA-10645,1323.9
AB-10015,341.96
AB-10060,4355.168
AB-10105,9892.74
AB-10150,479.97
AB-10165,304.23
AB-10255,479.984
